# SimpleBaseline Pose Estimation 구현

이 노트북에서는 SimpleBaseline의 구조를 이해하고,
ResNet backbone과 deconvolution module을 이용해
pose estimation 모델을 구현합니다.

In [2]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)

device: cpu


In [3]:
# 첫 번째 deconvolution 블록
# 입력 feature map의 채널 수 256을 유지하면서
# stride=2를 사용해 가로와 세로 크기를 각각 2배로 키웁니다.
upconv1 = nn.ConvTranspose2d(
    256,              # 입력 채널 수
    256,              # 출력 채널 수
    kernel_size=4,    # 4x4 커널
    stride=2,         # 해상도를 2배로 증가
    padding=1         # 출력 크기를 맞추기 위한 패딩
)

# deconvolution 결과를 정규화해 학습을 안정화합니다.
bn1 = nn.BatchNorm2d(256)

# 음수 값을 0으로 바꾸는 활성화 함수입니다.
relu1 = nn.ReLU()


# 두 번째 deconvolution 블록
# 첫 번째 블록에서 커진 feature map을 다시 2배로 키웁니다.
upconv2 = nn.ConvTranspose2d(
    256,
    256,
    kernel_size=4,
    stride=2,
    padding=1
)

bn2 = nn.BatchNorm2d(256)
relu2 = nn.ReLU()


# 세 번째 deconvolution 블록
# feature map의 해상도를 한 번 더 2배로 키웁니다.
upconv3 = nn.ConvTranspose2d(
    256,
    256,
    kernel_size=4,
    stride=2,
    padding=1
)

bn3 = nn.BatchNorm2d(256)
relu3 = nn.ReLU()

In [4]:
class DeconvLayer(nn.Module):
    def __init__(self, num_deconv_layers):
        super().__init__()

        # 여러 레이어를 순서대로 저장할 리스트
        layers = []

        # Deconv → BatchNorm → ReLU 블록을 지정한 횟수만큼 반복
        for _ in range(num_deconv_layers):
            layers.append(
                nn.ConvTranspose2d(
                    256,              # 입력 채널 수
                    256,              # 출력 채널 수
                    kernel_size=4,    # 4×4 커널
                    stride=2,         # 가로·세로 해상도를 2배로 증가
                    padding=1         # 출력 크기 조절
                )
            )

            # 출력값을 정규화해 학습을 안정화
            layers.append(nn.BatchNorm2d(256))

            # 비선형성을 추가하는 활성화 함수
            layers.append(nn.ReLU())

        # 리스트에 담긴 레이어들을 순서대로 실행하도록 묶기
        self.deconv_layers = nn.Sequential(*layers)

    def forward(self, x):
        # 입력 feature map을 deconv 블록들에 통과시킴
        return self.deconv_layers(x)


# Deconv 블록을 3개 가진 모듈 생성
upconv = DeconvLayer(3)

print(upconv)

DeconvLayer(
  (deconv_layers): Sequential(
    (0): ConvTranspose2d(256, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): ConvTranspose2d(256, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): ConvTranspose2d(256, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
)


In [5]:
# 256채널의 feature map을
# COCO 데이터셋의 17개 관절 heatmap으로 변환하는 마지막 출력 레이어
final_layer = nn.Conv2d(
    256,          # 입력 채널 수
    17,           # 출력 채널 수: 관절 17개
    kernel_size=1,
    padding=0
)

print(final_layer)

Conv2d(256, 17, kernel_size=(1, 1), stride=(1, 1))


In [7]:
class DeconvLayer(nn.Module):
    def __init__(self, in_channels=2048, num_deconv_layers=3):
        super().__init__()

        layers = []
        current_channels = in_channels

        # Deconv → BatchNorm → ReLU 블록을 3번 반복
        for _ in range(num_deconv_layers):
            layers.append(
                nn.ConvTranspose2d(
                    current_channels,  # 첫 블록은 2048, 이후에는 256
                    256,
                    kernel_size=4,
                    stride=2,
                    padding=1
                )
            )
            layers.append(nn.BatchNorm2d(256))
            layers.append(nn.ReLU(inplace=True))

            current_channels = 256

        self.deconv_layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.deconv_layers(x)


class PoseEstimationModel(nn.Module):
    def __init__(self):
        super().__init__()

        # ImageNet 사전학습 ResNet-50
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        # 평균 풀링과 분류층을 제외한 특징 추출 부분
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        # ResNet 출력 2048채널을 첫 deconv에서 256채널로 변환
        self.deconv = DeconvLayer(
            in_channels=2048,
            num_deconv_layers=3
        )

        # 17개 관절 heatmap 출력
        self.final_layer = nn.Conv2d(
            256,
            17,
            kernel_size=1
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.deconv(x)
        x = self.final_layer(x)
        return x


model = PoseEstimationModel().to(device)
print(model)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/jovyan/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 84.3MB/s]


PoseEstimationModel(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0)

In [8]:
# 배치 크기 1, RGB 3채널, 높이 256, 너비 192의 가상 이미지 생성
dummy_input = torch.randn(1, 3, 256, 192).to(device)

# 추론 모드로 변경
model.eval()

# gradient 계산 없이 출력 확인
with torch.no_grad():
    output = model(dummy_input)

print("입력 크기:", dummy_input.shape)
print("출력 크기:", output.shape)

입력 크기: torch.Size([1, 3, 256, 192])
출력 크기: torch.Size([1, 17, 64, 48])


## 출력 크기 확인

입력 이미지의 크기는 `256×192`이며, ResNet-50 backbone을 통과한 뒤
3개의 deconvolution layer에서 해상도가 단계적으로 2배씩 증가합니다.

최종 출력 shape은 다음과 같습니다.

- Batch size: 1
- 관절 수: 17
- Heatmap 높이: 64
- Heatmap 너비: 48

따라서 최종 출력 shape은 `[1, 17, 64, 48]`입니다.

In [9]:
%%time

# 값이 모두 0인 가상 입력 이미지 생성
# shape: [배치 크기, RGB 채널, 높이, 너비]
np_input = np.zeros((1, 3, 256, 192), dtype=np.float32)

# NumPy 배열을 PyTorch tensor로 변환
torch_input = torch.tensor(np_input, dtype=torch.float32)

# 사용 가능한 장치 선택
# 현재 환경에서는 CPU가 선택됨
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 입력 tensor와 모델을 같은 장치로 이동
torch_input = torch_input.to(device)
model = model.to(device)

# 추론 모드로 설정
model.eval()

print("input shape")
print(torch_input.shape)
print("\n")

# 추론 과정에서는 gradient를 계산하지 않음
with torch.no_grad():
    torch_output = model(torch_input)

print("output shape")
print(torch_output.shape)

# 출력 heatmap의 일부 값 확인
# 첫 번째 이미지에서 앞의 10개 관절,
# 세로·가로 각각 10칸의 값만 출력
print(torch_output[0, :10, :10, :10])

input shape
torch.Size([1, 3, 256, 192])


output shape
torch.Size([1, 17, 64, 48])
tensor([[[ 4.8895e-02,  5.0066e-02,  4.9461e-02,  4.9831e-02,  4.9493e-02,
           4.6976e-02,  5.1321e-02,  5.1123e-02,  4.8460e-02,  4.8011e-02],
         [ 4.8067e-02,  4.8880e-02,  4.8479e-02,  5.3579e-02,  4.9770e-02,
           4.8350e-02,  5.2306e-02,  4.7686e-02,  4.9672e-02,  5.0666e-02],
         [ 4.8733e-02,  5.0655e-02,  5.1074e-02,  4.7574e-02,  4.7719e-02,
           4.8762e-02,  5.3638e-02,  4.7044e-02,  4.8476e-02,  4.8856e-02],
         [ 4.7017e-02,  4.8066e-02,  5.1332e-02,  4.4387e-02,  4.7095e-02,
           5.2207e-02,  5.0492e-02,  4.9507e-02,  4.8510e-02,  5.1518e-02],
         [ 4.9246e-02,  4.9018e-02,  5.0554e-02,  4.8816e-02,  5.3139e-02,
           4.6025e-02,  5.3943e-02,  4.4270e-02,  5.2747e-02,  4.9175e-02],
         [ 4.7908e-02,  4.7798e-02,  4.8700e-02,  5.0096e-02,  5.1067e-02,
           4.9893e-02,  4.6119e-02,  5.1908e-02,  4.6843e-02,  4.6305e-02],
         [

## 실행 시간 확인

CPU 환경에서 배치 크기 1의 입력 이미지 한 장을 추론한 결과,
실제 경과 시간(Wall time)은 약 485ms가 소요되었습니다.

이는 모델의 정상 동작과 출력 크기를 확인하기 위한 테스트이며,
학습 속도나 전체 데이터셋 처리 성능을 의미하지는 않습니다.